# Dự án cuối kỳ Học Máy: Xây dựng Mô hình Dự đoán Sinh viên Thôi học sớm (OULAD)

**Mô tả:** Dự đoán sớm (tại các mốc D=30, 60, 90 ngày kể từ ngày bắt đầu học phần)
liệu một sinh viên có thôi học (Withdrawn) hay không, chỉ dùng thông tin có trong cửa sổ [0, D].

**Phạm vi:** Chỉ dùng các thuật toán Học Máy cổ điển (Hồi quy logistic, KNN, Cây quyết định CART,
Naive Bayes, Máy vector hỗ trợ SVM, Rừng ngẫu nhiên), kết hợp PCA (phân tích thành phần chính)
và phân cụm K-mean. KHÔNG dùng deep learning / neural network.

**Bộ dữ liệu:** Open University Learning Analytics Dataset (OULAD), UCI ML Repository.

**Điểm khác biệt (đặc trưng nâng cao, đã kiểm chứng từ các kernel/paper OULAD):**
- Engagement theo từng loại hoạt động VLE (forumng, quiz, oucontent, resource, ...)
- Recency, Frequency (days_active), đều đặn (regularity), độ đa dạng (diversity)
- Velocity: xu hướng click nửa sau so với nửa đầu cửa sổ
- Assessment nâng cao: điểm có trọng số, xu hướng điểm theo thời gian, điểm so với cohort
- Tương quan cohort: click/ngày hoạt động chuẩn hóa theo trung vị module-presentation
- Registration gap + cột đăng ký muộn; feature interaction (IMD×click, disability×active)
- Nhãn cụm K-mean làm đặc trưng bổ sung (stacking nhẹ, vẫn thuần ML)

## 1. Thiết lập môi trường và tải dữ liệu

Cell dưới đây tự tải dữ liệu từ UCI (chạy được trên Google Colab). Nếu dữ liệu đã có sẵn thì bỏ qua.

In [3]:
import os
import json
import zipfile
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, silhouette_score)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

# --- Các đường dẫn ---
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "dataset")
ART_DIR = os.path.join(BASE_DIR, "artifacts")
CACHE_DIR = os.path.join(ART_DIR, "cache")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

URL = "https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"

def tai_du_lieu():
    """Tải và giải nén OULAD nếu chưa có."""
    csv_can_thiet = ["studentInfo.csv", "studentRegistration.csv", "assessments.csv",
                     "studentAssessment.csv", "studentVle.csv", "vle.csv", "courses.csv"]
    if all(os.path.exists(os.path.join(DATA_DIR, f)) for f in csv_can_thiet):
        print("[OK] Dữ liệu đã có sẵn tại", DATA_DIR)
        return
    print("[*] Đang tải dữ liệu OULAD từ UCI...")
    zip_path = os.path.join(BASE_DIR, "data", "oulad.zip")
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print("[OK] Đã giải nén dữ liệu.")

tai_du_lieu()

[OK] Dữ liệu đã có sẵn tại e:\CaNhan\VoNhatHuy_Namba\HOCMAY\Machine learning\FINALEX\dataset


## 2. Đọc và tiền xử lý thô các bảng dữ liệu

`studentVle.csv` rất lớn (~10.6M dòng). Chúng ta chỉ cần hoạt động trong cửa sổ [0, 90] ngày,
nên lọc ngay từ lúc đọc theo chunk để tiết kiệm bộ nhớ.

In [4]:
print("[*] Đọc các bảng nhỏ...")
courses = pd.read_csv(os.path.join(DATA_DIR, "courses.csv"))
assessments = pd.read_csv(os.path.join(DATA_DIR, "assessments.csv"),
                          dtype={"code_module": "category", "code_presentation": "category"})
studentInfo = pd.read_csv(os.path.join(DATA_DIR, "studentInfo.csv"),
                          dtype={"code_module": "category", "code_presentation": "category",
                                 "gender": "category", "region": "category",
                                 "highest_education": "category", "imd_band": "category",
                                 "age_band": "category", "disability": "category",
                                 "final_result": "category"})
studentRegistration = pd.read_csv(os.path.join(DATA_DIR, "studentRegistration.csv"))
studentAssessment = pd.read_csv(os.path.join(DATA_DIR, "studentAssessment.csv"))
vle = pd.read_csv(os.path.join(DATA_DIR, "vle.csv"),
                  dtype={"code_module": "category", "code_presentation": "category",
                         "activity_type": "category"})

assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")
studentAssessment["score"] = pd.to_numeric(studentAssessment["score"], errors="coerce")
studentAssessment["date_submitted"] = pd.to_numeric(studentAssessment["date_submitted"], errors="coerce")
studentRegistration["date_registration"] = pd.to_numeric(studentRegistration["date_registration"], errors="coerce")
studentRegistration["date_unregistration"] = pd.to_numeric(studentRegistration["date_unregistration"], errors="coerce")

KEYS = ["code_module", "code_presentation", "id_student"]
print(f"[OK] studentInfo: {len(studentInfo)} dòng (mỗi dòng = 1 sinh viên trong 1 học phần)")

[*] Đọc các bảng nhỏ...
[OK] studentInfo: 32593 dòng (mỗi dòng = 1 sinh viên trong 1 học phần)


### 2.1 Xử lý bảng `studentVle` (clickstream)

Đọc theo từng chunk, chỉ giữ lại các bản ghi có `0 <= date <= 90`.
Giữ lại cả `activity_type` gốc để xây dựng đặc trưng engagement theo loại hoạt động.

In [5]:
NHOM_HOAT_DONG = {
    "forumng": "forum",
    "quiz": "quiz", "externalquiz": "quiz", "questionnaire": "quiz",
    "resource": "content", "oucontent": "content", "page": "content",
    "subpage": "content", "url": "content", "folder": "content",
    "glossary": "content", "dataplus": "content", "htmlactivity": "content",
    "repeatactivity": "content", "dualpane": "content", "sharedsubpage": "content",
    "homepage": "content", "ouwiki": "content",
    "oucollaborate": "collab", "workshop": "collab", "imcitutorial": "collab",
    "ouelluminate": "collab",
}
vle["nhom_hoat_dong"] = vle["activity_type"].astype(str).map(NHOM_HOAT_DONG).fillna("other")
site_to_nhom = dict(zip(vle["id_site"], vle["nhom_hoat_dong"]))
site_to_activity = dict(zip(vle["id_site"], vle["activity_type"].astype(str)))

# 16 loại hoạt động phổ biến nhất (theo số site), còn lại gom vào other_activity
ACTIVITY_TYPES = ["resource", "subpage", "oucontent", "url", "forumng", "quiz", "page",
                  "oucollaborate", "questionnaire", "ouwiki", "dataplus", "externalquiz",
                  "homepage", "glossary", "ouelluminate", "dualpane"]

print("[*] Đọc studentVle theo chunk, lọc 0 <= date <= 90...")
chunks = []
for chunk in pd.read_csv(os.path.join(DATA_DIR, "studentVle.csv"), chunksize=2_000_000,
                         dtype={"code_module": "category", "code_presentation": "category",
                                "id_student": "int32", "id_site": "int32",
                                "sum_click": "int32"}):
    chunk["date"] = pd.to_numeric(chunk["date"], errors="coerce")
    chunk = chunk[(chunk["date"] >= 0) & (chunk["date"] <= 90)]
    chunk["nhom_hoat_dong"] = chunk["id_site"].map(site_to_nhom).fillna("other")
    chunk["activity_type"] = chunk["id_site"].map(site_to_activity).fillna("other")
    chunks.append(chunk[["code_module", "code_presentation", "id_student", "id_site",
                         "date", "sum_click", "nhom_hoat_dong", "activity_type"]])
vle_small = pd.concat(chunks, ignore_index=True)
del chunks
print(f"[OK] Sau lọc cửa sổ [0,90]: {len(vle_small):,} dòng (giảm từ ~10.6M)")

[*] Đọc studentVle theo chunk, lọc 0 <= date <= 90...
[OK] Sau lọc cửa sổ [0,90]: 4,797,890 dòng (giảm từ ~10.6M)


## 3. Xây dựng bộ dữ liệu theo từng mốc thời gian D = 30, 60, 90 ngày

**Cohort tại mốc D:** chỉ giữ sinh viên vẫn còn đăng ký tại ngày D
(`date_unregistration` là NaN HOẶC > D).
**Nhãn:** y=1 nếu `final_result == 'Withdrawn'`, ngược lại y=0.

In [6]:
def imd_to_ord(x):
    if pd.isna(x) or str(x).strip() == "?":
        return np.nan
    s = str(x).strip()
    first = int(s.split("-")[0].replace("%", ""))
    return first // 10

studentInfo["imd_band_ord"] = studentInfo["imd_band"].astype(str).map(imd_to_ord)
studentInfo["age_band_ord"] = studentInfo["age_band"].map({"0-35": 0, "35-55": 1, "55<=": 2})
studentInfo["highest_education_ord"] = studentInfo["highest_education"].map({
    "No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
    "HE Qualification": 3, "Post Graduate Qualification": 4})

COT_DEMO_SO = ["num_of_prev_attempts", "studied_credits", "date_registration",
               "imd_band_ord", "age_band_ord", "highest_education_ord", "late_registration"]
COT_ASSESS = ["n_assessments_due", "n_submitted", "submission_rate", "score_mean",
              "score_max", "score_min", "n_late", "score_trend_slope",
              "weighted_score_mean", "relative_score_z"]
COT_VLE_CO_BAN = ["total_clicks", "active_days", "distinct_sites", "avg_clicks_per_active_day",
                  "last_active_recency", "forum_clicks", "quiz_clicks", "content_clicks",
                  "collab_clicks", "other_clicks", "forum_share", "quiz_share",
                  "content_share", "has_vle_activity"]
COT_VLE_NANG_CAO = ["n_activity_types", "std_daily_clicks", "regularity_ratio",
                    "clicks_first_half", "clicks_second_half", "click_velocity",
                    "clicks_last_7d", "relative_clicks", "relative_active_days"] + \
                   [f"click_{a}" for a in ACTIVITY_TYPES] + ["other_activity_clicks"]
COT_VLE = COT_VLE_CO_BAN + COT_VLE_NANG_CAO
COT_INTERACT = ["imd_x_logclicks", "disability_x_active", "credits_x_clicks"]
COT_SO = COT_DEMO_SO + COT_ASSESS + COT_VLE + COT_INTERACT
COT_HANH_VI = COT_ASSESS + COT_VLE  # đặc trưng hành vi cho KMeans
COT_NOMINAL = ["gender", "disability", "code_module", "code_presentation"]


def dac_trung_assessment(D):
    """Đặc trưng assessment trong cửa sổ [0, D], gồm điểm có trọng số và xu hướng điểm."""
    due = assessments[assessments["date"] <= D][["id_assessment", "code_module",
                                                  "code_presentation", "date", "weight"]]
    n_due = due.groupby(["code_module", "code_presentation"]).size().rename("n_assessments_due").reset_index()
    sub = studentAssessment.merge(due, on="id_assessment", how="inner")
    sub = sub[(sub["date_submitted"] <= D) | (sub["is_banked"] == 1)]
    sub["late"] = ((sub["date_submitted"] > sub["date"]) & (sub["is_banked"] == 0)).astype(int)
    agg = sub.groupby(KEYS).agg(
        n_submitted=("id_assessment", "count"),
        n_late=("late", "sum"),
        score_mean=("score", "mean"),
        score_max=("score", "max"),
        score_min=("score", "min")).reset_index()
    # Xu hướng điểm theo thời gian (slope) — tính vectorized trên bài có điểm
    sub_s = sub.dropna(subset=["score"]).copy()
    sub_s["d2"] = sub_s["date"] ** 2
    sub_s["ds"] = sub_s["date"] * sub_s["score"]
    sub_s["ws"] = sub_s["score"] * sub_s["weight"]
    tr = sub_s.groupby(KEYS).agg(
        n_scored=("score", "count"),
        sum_d=("date", "sum"), sum_s=("score", "sum"),
        sum_d2=("d2", "sum"), sum_ds=("ds", "sum"),
        sum_w=("weight", "sum"), sum_ws=("ws", "sum")).reset_index()
    tr["score_trend_slope"] = np.where(
        tr["n_scored"] >= 2,
        (tr["n_scored"] * tr["sum_ds"] - tr["sum_d"] * tr["sum_s"]) /
        (tr["n_scored"] * tr["sum_d2"] - tr["sum_d"] ** 2 + 1e-9), 0.0)
    tr["weighted_score_mean"] = tr["sum_ws"] / (tr["sum_w"] + 1e-9)
    agg = agg.merge(tr[KEYS + ["score_trend_slope", "weighted_score_mean"]], on=KEYS, how="left")
    return n_due, agg


def dac_trung_vle(D):
    """Đặc trưng clickstream nâng cao trong cửa sổ [0, D]."""
    df = vle_small[vle_small["date"] <= D]
    g = df.groupby(KEYS).agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        distinct_sites=("id_site", "nunique"),
        last_active=("date", "max"),
        n_activity_types=("activity_type", "nunique")).reset_index()
    g["avg_clicks_per_active_day"] = g["total_clicks"] / g["active_days"].clip(lower=1)
    g["last_active_recency"] = D - g["last_active"]
    g["regularity_ratio"] = g["active_days"] / D

    # Click theo nhóm hoạt động
    pivot_nhom = df.groupby(KEYS + ["nhom_hoat_dong"])["sum_click"].sum().unstack(fill_value=0).reset_index()
    for c in ["forum", "quiz", "content", "collab", "other"]:
        if c not in pivot_nhom.columns:
            pivot_nhom[c] = 0
        pivot_nhom.rename(columns={c: f"{c}_clicks"}, inplace=True)
    g = g.merge(pivot_nhom, on=KEYS, how="left").fillna(0)
    tong = g["total_clicks"].clip(lower=1)
    g["forum_share"] = g["forum_clicks"] / tong
    g["quiz_share"] = g["quiz_clicks"] / tong
    g["content_share"] = g["content_clicks"] / tong

    # Click theo từng loại hoạt động (engagement chi tiết)
    pivot_at = df.groupby(KEYS + ["activity_type"])["sum_click"].sum().unstack(fill_value=0).reset_index()
    at_cols = [c for c in pivot_at.columns if c not in KEYS]
    top_present = [a for a in ACTIVITY_TYPES if a in at_cols]
    other_at = [c for c in at_cols if c not in ACTIVITY_TYPES]
    pivot_at["other_activity_clicks"] = pivot_at[other_at].sum(axis=1) if other_at else 0
    g = g.merge(pivot_at[KEYS + top_present + ["other_activity_clicks"]], on=KEYS, how="left").fillna(0)
    for a in ACTIVITY_TYPES:
        if a in g.columns:
            g.rename(columns={a: f"click_{a}"}, inplace=True)
        else:
            g[f"click_{a}"] = 0

    # Độ biến động daily click (std)
    daily = df.groupby(KEYS + ["date"])["sum_click"].sum().reset_index()
    std_d = daily.groupby(KEYS)["sum_click"].std().rename("std_daily_clicks").reset_index()
    g = g.merge(std_d, on=KEYS, how="left")
    g["std_daily_clicks"] = g["std_daily_clicks"].fillna(0)

    # Velocity: nửa đầu vs nửa sau cửa sổ; click 7 ngày cuối (recency intensity)
    half = D / 2.0
    fh = df[df["date"] <= half].groupby(KEYS)["sum_click"].sum().rename("clicks_first_half").reset_index()
    sh = df[df["date"] > half].groupby(KEYS)["sum_click"].sum().rename("clicks_second_half").reset_index()
    l7 = df[df["date"] > D - 7].groupby(KEYS)["sum_click"].sum().rename("clicks_last_7d").reset_index()
    g = g.merge(fh, on=KEYS, how="left").merge(sh, on=KEYS, how="left").merge(l7, on=KEYS, how="left").fillna(0)
    g["click_velocity"] = ((g["clicks_second_half"] - g["clicks_first_half"]) /
                           (g["clicks_first_half"] + g["clicks_second_half"] + 1.0))
    g["has_vle_activity"] = 1
    return g


def xay_bo_du_lieu(D):
    cache_file = os.path.join(CACHE_DIR, f"features_D{D}.pkl")
    if os.path.exists(cache_file):
        print(f"[D={D}] Tải đặc trưng từ cache...")
        return joblib.load(cache_file)

    reg = studentRegistration.copy()
    cohort = reg[(reg["date_unregistration"].isna()) | (reg["date_unregistration"] > D)]
    cohort = cohort[cohort["date_registration"] <= D]
    cohort = cohort[KEYS + ["date_registration"]]

    df = cohort.merge(studentInfo[KEYS + ["gender", "disability", "num_of_prev_attempts",
                                          "studied_credits", "imd_band_ord", "age_band_ord",
                                          "highest_education_ord", "final_result"]], on=KEYS, how="inner")
    df["y"] = (df["final_result"] == "Withdrawn").astype(int)
    df.drop(columns=["final_result"], inplace=True)
    df["late_registration"] = (df["date_registration"] > 0).astype(int)

    n_due, agg_sub = dac_trung_assessment(D)
    df = df.merge(n_due, on=["code_module", "code_presentation"], how="left")
    df["n_assessments_due"] = df["n_assessments_due"].fillna(0)
    df = df.merge(agg_sub, on=KEYS, how="left")
    for c in ["n_submitted", "n_late", "score_trend_slope"]:
        df[c] = df[c].fillna(0)
    for c in ["score_mean", "score_max", "score_min", "weighted_score_mean"]:
        df[c] = df[c].fillna(0)
    df["submission_rate"] = df["n_submitted"] / df["n_assessments_due"].clip(lower=1)

    fvle = dac_trung_vle(D)
    df = df.merge(fvle, on=KEYS, how="left")

    # Chuẩn hóa theo cohort (module-presentation): không dùng nhãn -> không rò rỉ
    for col, new in [("total_clicks", "relative_clicks"), ("active_days", "relative_active_days")]:
        med = df.groupby(["code_module", "code_presentation"])[col].transform("median")
        df[new] = df[col] / (med + 1e-9)
    mean_s = df.groupby(["code_module", "code_presentation"])["score_mean"].transform("mean")
    std_s = df.groupby(["code_module", "code_presentation"])["score_mean"].transform("std")
    df["relative_score_z"] = (df["score_mean"] - mean_s) / (std_s + 1e-9)

    # Điền missing cho tất cả đặc trưng số (đã tồn tại hết)
    for c in COT_SO:
        if c in df.columns:
            df[c] = df[c].fillna(0)
    df["last_active_recency"] = df["last_active_recency"].replace([np.inf, -np.inf], D).fillna(D)
    df["relative_score_z"] = df["relative_score_z"].fillna(0)

    # Feature interaction (hữu ích cho Logistic/SVM)
    df["imd_x_logclicks"] = df["imd_band_ord"] * np.log1p(df["total_clicks"])
    df["disability_x_active"] = (df["disability"] == "Y").astype(int) * df["active_days"]
    df["credits_x_clicks"] = df["studied_credits"] * df["total_clicks"]
    for c in COT_INTERACT:
        df[c] = df[c].fillna(0)

    df = df.reset_index(drop=True)
    joblib.dump(df, cache_file)
    print(f"[D={D}] Xây dựng xong: {len(df)} mẫu, {len(COT_SO)} đặc trưng số, tỉ lệ thôi học = {df['y'].mean():.3f}")
    return df


DEADLINES = [30, 60, 90]
bo_du_lieu = {D: xay_bo_du_lieu(D) for D in DEADLINES}

[D=30] Xây dựng xong: 27443 mẫu, 60 đặc trưng số, tỉ lệ thôi học = 0.183
[D=60] Xây dựng xong: 26346 mẫu, 60 đặc trưng số, tỉ lệ thôi học = 0.149
[D=90] Xây dựng xong: 25551 mẫu, 60 đặc trưng số, tỉ lệ thôi học = 0.123


### 3.1 Biểu đồ cân bằng lớp theo từng mốc thời gian

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
counts = [(D, (bo_du_lieu[D]["y"] == 0).sum(), (bo_du_lieu[D]["y"] == 1).sum()) for D in DEADLINES]
cnt_df = pd.DataFrame(counts, columns=["D", "Không thôi (y=0)", "Thôi học (y=1)"]).set_index("D")
cnt_df.plot(kind="bar", ax=ax, color=["#4C9F70", "#E15759"])
ax.set_title("Số mẫu mỗi lớp theo mốc thời gian D (ngày)")
ax.set_xlabel("Mốc thời gian D (ngày)"); ax.set_ylabel("Số sinh viên - học phần")
ax.legend(title="Nhãn"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(ART_DIR, "class_balance.png"), dpi=120); plt.close()
print("[OK] Đã lưu class_balance.png")

[OK] Đã lưu class_balance.png


## 4. Tiền xử lý, tách train/test theo sinh viên, PCA và K-mean

In [8]:
class ClusterAdder(BaseEstimator, TransformerMixin):
    """Thêm one-hot nhãn cụm (từ KMeans đã fit) vào ma trận đặc trưng."""
    def __init__(self, kmeans, behavior_idx, n_clusters):
        self.kmeans = kmeans; self.behavior_idx = behavior_idx; self.n_clusters = n_clusters
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = self.kmeans.predict(X[:, self.behavior_idx])
        return np.hstack([X, np.eye(self.n_clusters)[c]])


def tien_xu_ly_va_tach(df, D):
    X = df[COT_SO + COT_NOMINAL].copy()
    y = df["y"].values
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df["id_student"]))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    so_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    nom_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    pre = ColumnTransformer([("so", so_pipe, COT_SO), ("nom", nom_pipe, COT_NOMINAL)])
    Xtr = pre.fit_transform(X_train); Xte = pre.transform(X_test)
    ten_nom = list(pre.named_transformers_["nom"]["onehot"].get_feature_names_out(COT_NOMINAL))
    ten_dac_trung = COT_SO + ten_nom
    behavior_idx = [COT_SO.index(c) for c in COT_HANH_VI]
    print(f"[D={D}] train={len(y_train)} (thôi={y_train.mean():.3f}), test={len(y_test)} "
          f"(thôi={y_test.mean():.3f}), số đặc trưng={Xtr.shape[1]}")
    return Xtr, Xte, y_train, y_test, pre, ten_dac_trung, behavior_idx


X90_tr, X90_te, y90_tr, y90_te, pre90, ten90, bidx90 = tien_xu_ly_va_tach(bo_du_lieu[90], 90)

pca_full = PCA().fit(X90_tr)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n95 = int(np.searchsorted(cumvar, 0.95) + 1)
print(f"[PCA] Số component giữ 95% phương sai: {n95} / {X90_tr.shape[1]}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", color="#4E79A7")
ax.axhline(0.95, color="#E15759", linestyle="--", label="95% phương sai")
ax.axvline(n95, color="#59A14F", linestyle=":", label=f"n={n95}")
ax.set_xlabel("Số thành phần chính"); ax.set_ylabel("Tỉ lệ phương sai tích lũy")
ax.set_title("PCA: Đồ thị phương sai tích lũy (D=90)"); ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(ART_DIR, "pca_variance.png"), dpi=120); plt.close()

pca2 = PCA(n_components=2, random_state=42).fit(X90_tr)
Z = pca2.transform(X90_tr)
rng = np.random.RandomState(42)
idx_plot = rng.choice(len(Z), size=min(5000, len(Z)), replace=False)
fig, ax = plt.subplots(figsize=(8, 6))
for lab, col, name in [(0, "#4C9F70", "Không thôi"), (1, "#E15759", "Thôi học")]:
    m = y90_tr[idx_plot] == lab
    ax.scatter(Z[idx_plot][m, 0], Z[idx_plot][m, 1], s=8, alpha=0.5, c=col, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Trực quan 2D PCA tô màu theo nhãn (D=90, lấy mẫu 5000)"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(ART_DIR, "pca_2d.png"), dpi=120); plt.close()
print("[OK] Đã lưu pca_variance.png, pca_2d.png")

[D=90] train=20439 (thôi=0.120), test=5112 (thôi=0.132), số đặc trưng=75
[PCA] Số component giữ 95% phương sai: 36 / 75
[OK] Đã lưu pca_variance.png, pca_2d.png


### 4.1 Phân cụm K-mean trên đặc trưng hành vi (D=90) — elbow + silhouette

In [9]:
hv_scaler = StandardScaler().fit(X90_tr[:, bidx90])
Hv_tr = hv_scaler.transform(X90_tr[:, bidx90])
ks = list(range(2, 11)); inercias, sils = [], []
sub_sil = rng.choice(len(Hv_tr), size=min(8000, len(Hv_tr)), replace=False)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(Hv_tr)
    inercias.append(km.inertia_)
    sils.append(silhouette_score(Hv_tr[sub_sil], lab[sub_sil], sample_size=4000, random_state=42))
    print(f"[KMeans] k={k}: inertia={km.inertia_:.0f}, silhouette={sils[-1]:.4f}")
k_tot_nhat = ks[int(np.argmax(sils))]
print(f"[KMeans] Chọn k={k_tot_nhat} (silhouette cao nhất)")

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(ks, inercias, "o-", color="#4E79A7", label="Inertia (elbow)")
ax1.set_xlabel("Số cụm k"); ax1.set_ylabel("Inertia", color="#4E79A7")
ax2 = ax1.twinx()
ax2.plot(ks, sils, "s--", color="#E15759", label="Silhouette")
ax2.set_ylabel("Silhouette score", color="#E15759")
ax2.axvline(k_tot_nhat, color="#59A14F", linestyle=":", label=f"k chọn={k_tot_nhat}")
plt.title("KMeans: Elbow + Silhouette theo k (đặc trưng hành vi, D=90)")
fig.tight_layout(); plt.savefig(os.path.join(ART_DIR, "kmeans_elbow.png"), dpi=120); plt.close()

km_final = KMeans(n_clusters=k_tot_nhat, random_state=42, n_init=10).fit(Hv_tr)
joblib.dump(km_final, os.path.join(ART_DIR, "kmeans.joblib"))
cl_tr = km_final.predict(Hv_tr)
prof = pd.DataFrame(Hv_tr, columns=COT_HANH_VI); prof["cluster"] = cl_tr; prof["y"] = y90_tr
profile = prof.groupby("cluster").agg(**{c: (c, "mean") for c in COT_HANH_VI},
                                      n_students=("y", "count"),
                                      ti_le_thoi_hoc=("y", "mean")).reset_index()
profile.to_csv(os.path.join(ART_DIR, "cluster_profiles.csv"), index=False)

fig, ax = plt.subplots(figsize=(12, 6))
show_cols = [c for c in COT_HANH_VI if not c.startswith("click_")][:14]
sns.heatmap(profile.set_index("cluster")[show_cols].T, cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "Giá trị trung bình (đã chuẩn hóa)"})
ax.set_title(f"Hồ sơ {k_tot_nhat} cụm KMeans (D=90). " +
             " | ".join([f"Cụm {r['cluster']}: {r['ti_le_thoi_hoc']:.0%} thôi" for _, r in profile.iterrows()]))
ax.set_xlabel("Cụm"); ax.set_ylabel("Đặc trưng hành vi"); plt.tight_layout()
plt.savefig(os.path.join(ART_DIR, "cluster_profile.png"), dpi=120); plt.close()
print("[OK] Đã lưu kmeans_elbow.png, cluster_profile.png, kmeans.joblib, cluster_profiles.csv")

[KMeans] k=2: inertia=823677, silhouette=0.3022
[KMeans] k=3: inertia=731850, silhouette=0.2882
[KMeans] k=4: inertia=682229, silhouette=0.1685
[KMeans] k=5: inertia=650508, silhouette=0.1189
[KMeans] k=6: inertia=620732, silhouette=0.1203
[KMeans] k=7: inertia=595982, silhouette=0.1375
[KMeans] k=8: inertia=573369, silhouette=0.1375
[KMeans] k=9: inertia=553351, silhouette=0.1372
[KMeans] k=10: inertia=536705, silhouette=0.1506
[KMeans] Chọn k=2 (silhouette cao nhất)
[OK] Đã lưu kmeans_elbow.png, cluster_profile.png, kmeans.joblib, cluster_profiles.csv


## 5. Huấn luyện và đánh giá các mô hình phân lớp tại mỗi mốc D

Mô hình: Hồi quy logistic, KNN, Cây quyết định (CART), Gaussian Naive Bayes,
Máy vector hỗ trợ (SVC RBF, probability=True), Rừng ngẫu nhiên.
Thêm: Hồi quy logistic trên PCA, và Hồi quy logistic + SMOTE (đối chiếu mất cân bằng lớp).
Tối ưu nhẹ siêu tham số bằng GridSearchCV (cv=3).

In [10]:
def danh_gia_mo_hinh(y_true, y_pred, y_prob):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_prob),
            "confusion": confusion_matrix(y_true, y_pred).tolist()}


ket_qua = []; du_doan = {}; mo_hinh_tot = None; best_pipeline_obj = None
rf_best_for_importance = None; ten_dac_trung_for_importance = None

for D in DEADLINES:
    df = bo_du_lieu[D]
    Xtr, Xte, ytr, yte, pre, ten_dac_trung, behavior_idx = tien_xu_ly_va_tach(df, D)
    hv_sc = StandardScaler().fit(Xtr[:, behavior_idx])
    km_D = KMeans(n_clusters=k_tot_nhat, random_state=42, n_init=10).fit(hv_sc.transform(Xtr[:, behavior_idx]))
    cluster_adder = ClusterAdder(km_D, behavior_idx, k_tot_nhat)
    Xtr_ext = cluster_adder.transform(Xtr); Xte_ext = cluster_adder.transform(Xte)
    ten_cuoi = ten_dac_trung + [f"cluster_{i}" for i in range(k_tot_nhat)]

    if D == 90:
        with open(os.path.join(ART_DIR, "feature_names.txt"), "w") as f:
            f.write("\n".join(ten_cuoi))
        joblib.dump(pre, os.path.join(ART_DIR, "preprocessor.joblib"))

    du_doan[D] = {}; models = {}
    lr = GridSearchCV(LogisticRegression(class_weight="balanced", max_iter=3000, solver="lbfgs"),
                      {"C": [0.01, 0.1, 1.0]}, cv=3, scoring="roc_auc", n_jobs=-1).fit(Xtr_ext, ytr)
    models["Hồi quy logistic"] = lr.best_estimator_
    knn = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [15, 31], "weights": ["uniform", "distance"]},
                       cv=3, scoring="roc_auc", n_jobs=-1).fit(Xtr_ext, ytr)
    models["K láng giềng gần (KNN)"] = knn.best_estimator_
    dt = GridSearchCV(DecisionTreeClassifier(class_weight="balanced", random_state=42),
                      {"max_depth": [5, 10, None], "min_samples_leaf": [20]},
                      cv=3, scoring="roc_auc", n_jobs=-1).fit(Xtr_ext, ytr)
    models["Cây quyết định (CART)"] = dt.best_estimator_
    models["Gaussian Naive Bayes"] = GaussianNB().fit(Xtr_ext, ytr)
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
              probability=True, random_state=42).fit(Xtr_ext, ytr)
    models["Máy vector hỗ trợ (SVM)"] = svm
    rf = GridSearchCV(RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                             random_state=42, n_jobs=-1),
                      {"max_depth": [10, None], "min_samples_leaf": [5, 20]},
                      cv=3, scoring="roc_auc", n_jobs=1).fit(Xtr_ext, ytr)
    models["Rừng ngẫu nhiên"] = rf.best_estimator_
    if D == 90:
        rf_best_for_importance = rf.best_estimator_; ten_dac_trung_for_importance = ten_cuoi
    pcaD = PCA(n_components=min(n95, Xtr.shape[1]), random_state=42).fit(Xtr)
    lr_pca = LogisticRegression(class_weight="balanced", max_iter=3000).fit(pcaD.transform(Xtr), ytr)
    models["Hồi quy logistic (PCA)"] = ("pca", pcaD, lr_pca)
    Xs, ys = SMOTE(random_state=42).fit_resample(Xtr_ext, ytr)
    models["Hồi quy logistic + SMOTE"] = LogisticRegression(max_iter=3000).fit(Xs, ys)

    for ten, md in models.items():
        if isinstance(md, tuple) and md[0] == "pca":
            _, pca_m, lr_m = md
            y_prob = lr_m.predict_proba(pca_m.transform(Xte))[:, 1]; y_pred = lr_m.predict(pca_m.transform(Xte))
        else:
            y_prob = md.predict_proba(Xte_ext)[:, 1]; y_pred = md.predict(Xte_ext)
        m = danh_gia_mo_hinh(yte, y_pred, y_prob)
        tn, fp, fn, tp = np.array(m["confusion"]).ravel()
        ket_qua.append({"moc_D": D, "mo_hinh": ten, "accuracy": m["accuracy"],
                        "precision": m["precision"], "recall": m["recall"],
                        "f1": m["f1"], "roc_auc": m["roc_auc"], "tn": tn, "fp": fp, "fn": fn, "tp": tp})
        du_doan[D][ten] = (yte, y_prob)
        score = (m["f1"], m["roc_auc"])
        if mo_hinh_tot is None or score > (mo_hinh_tot[0], mo_hinh_tot[1]):
            mo_hinh_tot = (m["f1"], m["roc_auc"], D, ten)
            if ten == "Hồi quy logistic (PCA)":
                best_pipeline_obj = Pipeline([("pre", pre), ("pca", pcaD), ("clf", lr_m)])
            else:
                best_pipeline_obj = Pipeline([("pre", pre), ("cluster", cluster_adder), ("clf", md)])
    print(f"[D={D}] Đã huấn luyện & đánh giá xong {len(models)} mô hình.")

eval_df = pd.DataFrame(ket_qua)
eval_df.to_csv(os.path.join(ART_DIR, "evaluation_results.csv"), index=False)
print("[OK] Đã lưu evaluation_results.csv")
print(eval_df.sort_values(["moc_D", "f1"], ascending=[True, False]).round(3).to_string(index=False))

[D=30] train=21936 (thôi=0.183), test=5507 (thôi=0.184), số đặc trưng=75
[D=30] Đã huấn luyện & đánh giá xong 8 mô hình.
[D=60] train=21035 (thôi=0.150), test=5311 (thôi=0.147), số đặc trưng=75
[D=60] Đã huấn luyện & đánh giá xong 8 mô hình.
[D=90] train=20439 (thôi=0.120), test=5112 (thôi=0.132), số đặc trưng=75
[D=90] Đã huấn luyện & đánh giá xong 8 mô hình.
[OK] Đã lưu evaluation_results.csv
 moc_D                  mo_hinh  accuracy  precision  recall    f1  roc_auc   tn   fp  fn  tp
    30  Máy vector hỗ trợ (SVM)     0.701      0.332   0.613 0.431    0.716 3240 1252 393 622
    30         Hồi quy logistic     0.682      0.317   0.629 0.422    0.712 3119 1373 377 638
    30          Rừng ngẫu nhiên     0.762      0.380   0.455 0.414    0.719 3737  755 553 462
    30 Hồi quy logistic + SMOTE     0.670      0.306   0.625 0.411    0.704 3057 1435 381 634
    30   Hồi quy logistic (PCA)     0.672      0.305   0.606 0.405    0.703 3088 1404 400 615
    30    Cây quyết định (CART)     0.

## 6. Trực quan hóa đánh giá: ROC, ma trận nhầm lẫn, feature importance

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
colors = plt.cm.tab10(np.linspace(0, 1, 8))
for ax, D in zip(axes, DEADLINES):
    for i, (ten, (yte, yp)) in enumerate(du_doan[D].items()):
        fpr, tpr, _ = roc_curve(yte, yp); auc = roc_auc_score(yte, yp)
        ax.plot(fpr, tpr, color=colors[i], lw=1.5, label=f"{ten} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(f"Đường cong ROC — D={D} ngày")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig(os.path.join(ART_DIR, "roc_curves.png"), dpi=120); plt.close()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, D in zip(axes, DEADLINES):
    sub = eval_df[eval_df.moc_D == D].sort_values("f1", ascending=False).iloc[0]
    cm = np.array([[sub.tn, sub.fp], [sub.fn, sub.tp]])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Dự đoán 0", "Dự đoán 1"], yticklabels=["Thực 0", "Thực 1"])
    ax.set_title(f"{sub.mo_hinh}\nD={D} | F1={sub.f1:.3f} | Recall={sub.recall:.3f} | AUC={sub.roc_auc:.3f}")
plt.tight_layout(); plt.savefig(os.path.join(ART_DIR, "confusion_matrices.png"), dpi=120); plt.close()

if rf_best_for_importance is not None:
    imp = pd.Series(rf_best_for_importance.feature_importances_,
                    index=ten_dac_trung_for_importance).sort_values(ascending=False)
    top15 = imp.head(15)
    fig, ax = plt.subplots(figsize=(9, 7))
    top15[::-1].plot(kind="barh", ax=ax, color="#4E79A7")
    ax.set_title("Top 15 đặc trưng quan trọng nhất (Rừng ngẫu nhiên, D=90)")
    ax.set_xlabel("Feature importance"); plt.tight_layout()
    plt.savefig(os.path.join(ART_DIR, "feature_importance.png"), dpi=120); plt.close()
    top10 = imp.head(10).to_dict()
else:
    top10 = {}
print("[OK] Đã lưu roc_curves.png, confusion_matrices.png, feature_importance.png")

[OK] Đã lưu roc_curves.png, confusion_matrices.png, feature_importance.png


## 7. Benchmark công khai, lưu mô hình và tóm tắt kết quả

In [12]:
# Benchmark công khai trên OULAD (early dropout / at-risk prediction)
BENCHMARKS = [
    {"nguon": "MDPI Data 2026 — Explainable Early Warnings for Dropout (days 28/56/84, student-grouped)",
     "url": "https://www.mdpi.com/2306-5729/11/11/281",
     "chi_so": "LogReg ROC-AUC = 0.687/0.692/0.710; Random Forest ~0.710 tại day-84"},
    {"nguon": "Frontiers in Education 2024 — Analyzing click data with AI",
     "url": "https://www.frontiersin.org/journals/education/articles/10.3389/feduc.2024.1421479/full",
     "chi_so": "Random Forest accuracy = 78.68% (xác định sinh viên có nguy cơ thôi học)"},
    {"nguon": "Emerald AAOUJ 2026 — Predicting student success in open and distance learning",
     "url": "https://www.emerald.com/aaouj/article-split/doi/10.1108/AAOUJ-05-2026-0090/1401235/Predicting-student-success-in-open-and-distance",
     "chi_so": "Linear SVM macro-F1 = 0.406, Random Forest = 0.404; đặc trưng hành vi nâng macro-F1 +0.014"},
    {"nguon": "PeerJ Computer Science 2024 — Predicting student success in MOOCs (OULAD)",
     "url": "https://peerj.com/articles/cs-2221/",
     "chi_so": "Phân lớp 2 nhãn: LogReg accuracy=92.4%, RF=95.7% (cuối khóa, định nghĩa bài toán khác dễ hơn)"},
    {"nguon": "arXiv 2512.12493 — AI-Driven Early Warning Systems (Static Feature Dominance)",
     "url": "https://arxiv.org/html/2512.12493",
     "chi_so": "Đặc trưng demographic tĩnh chiếm 68% importance; sum_click chỉ 1.34%"},
    {"nguon": "Kuzilek, Hlosta, Zdrahal 2017 — OULAD original dataset paper",
     "url": "https://www.nature.com/articles/sdata2017171",
     "chi_so": "Bộ dữ liệu chuẩn: 32,593 đăng ký, 22 module-presentation, ~10.6M tương tác VLE"},
]
with open(os.path.join(ART_DIR, "benchmarks.json"), "w", encoding="utf-8") as f:
    json.dump(BENCHMARKS, f, ensure_ascii=False, indent=2)

joblib.dump(best_pipeline_obj, os.path.join(ART_DIR, "best_model.joblib"))
print(f"[OK] Mô hình tốt nhất: {mo_hinh_tot[3]} tại D={mo_hinh_tot[2]} "
      f"(F1={mo_hinh_tot[0]:.4f}, ROC-AUC={mo_hinh_tot[1]:.4f})")

with open(os.path.join(ART_DIR, "requirements.txt"), "w") as f:
    f.write("pandas\nnumpy\nscikit-learn\nimbalanced-learn\nmatplotlib\nseaborn\njoblib\nstreamlit\n")

# Đặc trưng nâng cao và mức đóng góp (importance từ RF D=90)
dac_trung_nang_cao = COT_VLE_NANG_CAO + COT_INTERACT + ["score_trend_slope", "weighted_score_mean",
                                                        "relative_score_z", "late_registration",
                                                        "relative_clicks", "relative_active_days"]
importance_nang_cao = {k: float(top10.get(k, 0.0)) for k in dac_trung_nang_cao}
if rf_best_for_importance is not None:
    full_imp = pd.Series(rf_best_for_importance.feature_importances_, index=ten_dac_trung_for_importance)
    importance_nang_cao = {k: float(full_imp.get(k, 0.0)) for k in dac_trung_nang_cao}

best_per_D = {str(D): eval_df[eval_df.moc_D == D].sort_values(["f1", "roc_auc"], ascending=False).iloc[0][
    ["mo_hinh", "accuracy", "precision", "recall", "f1", "roc_auc"]].to_dict() for D in DEADLINES}

summary = {
    "best_model": mo_hinh_tot[3], "best_deadline_D": int(mo_hinh_tot[2]),
    "best_f1": float(mo_hinh_tot[0]), "best_roc_auc": float(mo_hinh_tot[1]),
    "best_model_per_D": best_per_D,
    "metrics_by_best_model_per_D": {
        str(D): eval_df[(eval_df.moc_D == D) & (eval_df.mo_hinh == mo_hinh_tot[3])][
            ["accuracy", "precision", "recall", "f1", "roc_auc"]].to_dict("records") for D in DEADLINES},
    "top10_features": top10,
    "advanced_features_importance": importance_nang_cao,
    "benchmarks": BENCHMARKS,
    "class_counts_per_D": {str(D): {"0": int((bo_du_lieu[D].y == 0).sum()),
                                    "1": int((bo_du_lieu[D].y == 1).sum())} for D in DEADLINES},
    "dataset_size_per_D": {str(D): int(len(bo_du_lieu[D])) for D in DEADLINES},
    "n_clusters_kmeans": int(k_tot_nhat), "pca_n_components_95pct": int(n95),
    "notes": ("studentVle lọc còn cửa sổ [0,90] (~4.8M dòng); tách train/test theo id_student; "
              "class_weight='balanced'; SMOTE đối chiếu trên Hồi quy logistic; đặc trưng nâng cao: "
              "engagement theo activity_type, recency, regularity, diversity, velocity, score trend, "
              "cohort-relative normalization, feature interaction, KMeans cluster label. Không dùng deep learning.")
}
with open(os.path.join(ART_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("[OK] Đã lưu best_model.joblib, requirements.txt, summary.json, benchmarks.json")
print("[HOÀN TẤT] Tất cả artifacts đã được lưu tại", ART_DIR)

[OK] Mô hình tốt nhất: Máy vector hỗ trợ (SVM) tại D=30 (F1=0.4306, ROC-AUC=0.7162)
[OK] Đã lưu best_model.joblib, requirements.txt, summary.json, benchmarks.json
[HOÀN TẤT] Tất cả artifacts đã được lưu tại e:\CaNhan\VoNhatHuy_Namba\HOCMAY\Machine learning\FINALEX\artifacts


## 8. Nguồn tham khảo

1. OULAD dataset (UCI): https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset
2. Kuzilek, Hlosta, Zdrahal (2017), Scientific Data: https://www.nature.com/articles/sdata2017171
3. MDPI Data 2026 (early warning, days 28/56/84): https://www.mdpi.com/2306-5729/11/11/281
4. Frontiers in Education 2024 (click data, RF 78.68%): https://www.frontiersin.org/journals/education/articles/10.3389/feduc.2024.1421479/full
5. Emerald AAOUJ 2026 (engagement features, macro-F1): https://www.emerald.com/aaouj/article-split/doi/10.1108/AAOUJ-05-2026-0090/1401235/Predicting-student-success-in-open-and-distance
6. PeerJ CS 2024 (OULAD two-class): https://peerj.com/articles/cs-2221/
7. arXiv 2512.12493 (static feature dominance): https://arxiv.org/html/2512.12493
8. Computational Social Science Cookbook (sum_clicks = top feature): https://data-edu.github.io/computational-social-science-cookbook/chapter-6.html